Import Libraries

In [1]:
import torch
import evaluate
import os
import re
import pandas as pd
import numpy as np

from torch import nn
from torch.utils.data import Dataset, DataLoader

from datasets import load_dataset

from transformers import AutoConfig, AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
from sklearn.metrics import classification_report, accuracy_score, precision_recall_fscore_support

c:\Users\Weedguet\Documents\GitHub\mental_health_classifier\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Config

In [2]:
MODEL_NAME = "distilbert-base-uncased" 
MAX_LEN = 32
SEED = 42
EPOCHS = 4
np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
device

is_multilabel = False

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

macro_f1 = evaluate.load("f1")
micro_f1 = evaluate.load("f1")

# print(torch.__version__)
# print(torch.version.cuda)   # PyTorch CUDA version
# print(torch.cuda.is_available())

Toekenization setup

In [3]:
def tok_batch(texts):
    return tokenizer(
        texts, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt"
    )

Helper functions

In [4]:
def clean_text(text):
    text = re.sub(r"[^\w\s]", " ", text)
    text = re.sub(r"\b[a-zA-Z]\b", " ", text)
    text = re.sub(r"<[^>]*>", " ", text)
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text

def clean_data(data):
    new_data = data[['label', 'text']].copy()
    texts = new_data['text'].tolist()
    cleaned_texts = [clean_text(str(text)) for text in texts]
    new_data['text'] = cleaned_texts
    return new_data

Metrics

In [5]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if not is_multilabel:
        preds = np.argmax(logits, axis=1)
        return {
            "macro_f1": macro_f1.compute(predictions=preds, references=labels, average="macro")["f1"],
            "accuracy": (preds == labels).mean()
        }
    else:
        probs = 1/(1+np.exp(-logits))
        preds = (probs >= 0.5).astype(int)
        return {
            "macro_f1": macro_f1.compute(predictions=preds, references=labels, average="macro")["f1"],
            "micro_f1": micro_f1.compute(predictions=preds, references=labels, average="micro")["f1"]
        }

Inference on new text

In [6]:
def predict(texts, threshold=0.5):
    enc = tok(texts, padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(device)
    with torch.no_grad():
        logits = mdl(**enc).logits
    if not is_multilabel:
        ids = torch.argmax(logits, dim=-1).cpu().numpy().tolist()
        return [label_names[i] for i in ids]
    else:
        probs = torch.sigmoid(logits).cpu().numpy()
        pred_mat = (probs >= threshold).astype(int)
        return [[label_names[i] for i,v in enumerate(row) if v==1] for row in pred_mat]

Class for datasets setup

In [7]:
class TextDataset(Dataset):
    def __init__(self, df, is_multilabel=False):
        self.texts = df["text"].tolist()
        if not is_multilabel:
            self.labels = df["y"].tolist()
        else:
            self.labels = [mlb.transform([labs])[0] for labs in df["label"]]
        self.is_multilabel = is_multilabel
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = tokenizer(self.texts[idx], padding="max_length", truncation=True,
                        max_length=MAX_LEN, return_tensors="pt")
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if not self.is_multilabel:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        else:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

Handle class imbalance (Trainer with custom loss)

In [8]:
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, pos_weight=None, is_multilabel=False, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.pos_weight = pos_weight
        self.is_multilabel = is_multilabel

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels", None)
        outputs = model(**inputs)
        logits = outputs.logits

        if labels is None:
            loss = outputs.loss if hasattr(outputs, "loss") else None
            return (loss, outputs) if return_outputs else loss

        labels = labels.to(logits.device)

        if not self.is_multilabel:
            if self.class_weights is None:
                loss_fct = nn.CrossEntropyLoss()
            else:
                loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(logits.device))
            loss = loss_fct(logits, labels)
        else:
            if self.pos_weight is None:
                loss_fct = nn.BCEWithLogitsLoss()
            else:
                loss_fct = nn.BCEWithLogitsLoss(pos_weight=self.pos_weight.to(logits.device))
            loss = loss_fct(logits, labels.type_as(logits))

        return (loss, outputs) if return_outputs else loss

In [9]:
def filter_by_label_keywords(df, keyword_dict, text_col="body"):
    keep_mask = []
    for _, row in df.iterrows():
        sub = row["label"]
        text = row[text_col].lower()
        matched = False

        if sub not in keyword_dict:
            matched = True  # keep everything from these subs
        elif sub in keyword_dict:
            for kw in keyword_dict[sub]:
                if re.search(rf"\b{kw}\b", text):  # word boundary match
                    matched = True
                    break
        keep_mask.append(matched)
    return df[keep_mask]

Load data

In [10]:
df = pd.read_csv("mentalhealth_reddit_posts_ds.csv")

df.rename(columns={"subreddit": "label"}, inplace=True)
df.rename(columns={"selftext": "text"}, inplace=True)

df = clean_data(df)

In [11]:
df['label'].value_counts()

label
AskReddit               5192
MentalHealth            3964
depression              3882
Anxiety                 3833
CPTSD                   3764
stopdrinking            3760
ADHD                    3696
BPD                     3690
OCD                     3636
bipolar                 2962
bipolar2                2816
BipolarReddit           2366
Anxietyhelp             2360
insomnia                2327
addiction               2282
GetDisciplined          2216
BipolarSOs              2111
NPD                     2108
dataisbeautiful         2011
Anger                   2003
hoarding                1972
ADHDers                 1957
Traumatoolbox           1920
aspd                    1421
personalitydisorders    1301
sleepdisorders          1297
ODD                      394
Name: count, dtype: int64

In [12]:
keyword_dict = {
    "Anxiety": ["anxiety", "panic attack", "social anxiety", "gad", "worry"],
    "Anxietyhelp": ["anxiety", "panic attack", "nervous", "restless", "fear"],
    "depression": ["depression", "depressed", "mdd", "hopeless", "sadness"],
    "MentalHealth": ["depression", "mental health", "therapy", "suicidal", "low mood"],
    "stopdrinking": ["alcohol", "sobriety", "relapse", "drinking", "quit"],
    "addiction": ["addiction", "withdrawal", "sobriety", "craving", "detox"],
    "CPTSD": ["ptsd", "trauma", "flashback", "cptsd", "nightmare"],
    "Traumatoolbox": ["trauma", "stress", "hypervigilance", "dissociation", "trigger"],
    "ADHD": ["adhd", "attention deficit", "hyperactive", "impulsive", "focus"],
    "ADHDers": ["adhd", "adhders", "inattentive", "executive function", "adhd life"],
    "bipolar": ["bipolar", "mania", "manic", "hypomania", "mood swing"],
    "bipolar2": ["bipolar ii", "hypomania", "depressive episode", "cycling"],
    "BPD": ["bpd", "borderline", "fear of abandonment", "splitting", "unstable"],
    "personalitydisorders": ["personality disorder", "narcissistic", "antisocial", "histrionic"],
    "OCD": ["ocd", "obsession", "compulsion", "ritual", "intrusive thoughts"],
    "hoarding": ["hoard", "clutter", "collecting", "can’t throw away"],
    "ODD": ["odd", "oppositional", "defiant", "authority", "tantrum"],
    "Anger": ["anger", "rage", "outburst", "impulse", "irritability"],
    "insomnia": ["insomnia", "sleep", "can’t sleep", "awake", "restless"],
    "sleepdisorders": ["narcolepsy", "sleep apnea", "parasomnia", "night terror"],
    "aspd": ["aspd", "antisocial", "sociopath", "psychopath", "lack of empathy"],
    "BipolarReddit": ["bipolar", "mania", "manic", "hypomania", "mood swing"],
    "BipolarSOs": ["bipolar", "partner", "relationship", "support", "caregiver"],
    "NPD": ["npd", "narcissist", "narcissistic", "grandiosity", "lack of empathy"],
    "GetDisciplined": ["discipline", "self control", "motivation", "habit", "impulse"],
}

In [13]:
df = filter_by_label_keywords(df, keyword_dict, text_col="text")

df['label'].value_counts()

label
AskReddit               5192
stopdrinking            2394
Anxiety                 2308
ADHD                    2068
dataisbeautiful         2011
insomnia                1991
OCD                     1923
BPD                     1580
CPTSD                   1378
depression              1354
BipolarSOs              1266
bipolar                 1216
MentalHealth            1208
Anger                   1165
BipolarReddit           1131
NPD                      947
Anxietyhelp              909
GetDisciplined           899
addiction                810
ADHDers                  790
Traumatoolbox            755
aspd                     566
hoarding                 436
personalitydisorders     393
bipolar2                 338
sleepdisorders           180
ODD                       10
Name: count, dtype: int64

In [14]:
label_mapping = {
    "Anxiety": "anxiety_disorder",
    "Anxietyhelp": "anxiety_disorder",
    "depression": "depression_disorder",
    "MentalHealth": "depression_disorder",
    "stopdrinking": "addiction_disorder",
    "addiction": "addiction_disorder",
    "CPTSD": "stress_disorder",
    "Traumatoolbox": "stress_disorder",
    "ADHD": "neurodevelopmental_disorder",
    "ADHDers": "neurodevelopmental_disorder",
    "bipolar": "bipolar_disorder",
    "bipolar2": "bipolar_disorder",
    "BipolarReddit": "bipolar_disorder",
    "BipolarSOs": "bipolar_disorder",
    "BPD": "personality_disorder",
    "personalitydisorders": "personality_disorder",
    "NPD": "personality_disorder",
    "OCD": "obsession_compulsion_disorder",
    "hoarding": "obsession_compulsion_disorder",
    "ODD": "conduct_disorder",
    "Anger": "conduct_disorder",
    "aspd": "conduct_disorder",
    "GetDisciplined": "conduct_disorder",
    "insomnia": "sleep_disorder",
    "sleepdisorders": "sleep_disorder",
    "AskReddit": "no_disorder",
    "dataisbeautiful": "no_disorder"
}

df['label'] = df['label'].replace(label_mapping)

df['label'].value_counts()

label
no_disorder                      7203
bipolar_disorder                 3951
anxiety_disorder                 3217
addiction_disorder               3204
personality_disorder             2920
neurodevelopmental_disorder      2858
conduct_disorder                 2640
depression_disorder              2562
obsession_compulsion_disorder    2359
sleep_disorder                   2171
stress_disorder                  2133
Name: count, dtype: int64

In [15]:
# df = (
#     df.groupby("label", group_keys=False)
#       .head(2000)
# )

df = df.reset_index(drop=True)

df['label'].value_counts()

label
no_disorder                      7203
bipolar_disorder                 3951
anxiety_disorder                 3217
addiction_disorder               3204
personality_disorder             2920
neurodevelopmental_disorder      2858
conduct_disorder                 2640
depression_disorder              2562
obsession_compulsion_disorder    2359
sleep_disorder                   2171
stress_disorder                  2133
Name: count, dtype: int64

Choose label mode (multi-class OR multi-label)

In [16]:
if not is_multilabel:
    enc = LabelEncoder()
    df["y"] = enc.fit_transform(df["label"])
    label_names = list(enc.classes_)
    num_labels = len(label_names)
    class_counts = df["y"].value_counts().sort_index().to_numpy()
    class_weights = (class_counts.sum() / (num_labels * class_counts))
    class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

    pos_weight = torch.ones(num_labels, dtype=torch.float32).to(device)
else:
    # Ensure label is a list
    if df["label"].dtype != object or not isinstance(df["label"].iloc[0], (list, tuple)):
        raise ValueError("For multi-label, make sure df['label'] is a list of strings per row.")
    mlb = MultiLabelBinarizer()
    Y = mlb.fit_transform(df["label"])
    label_names = list(mlb.classes_)
    num_labels = len(label_names)
    # Positive class weights (for BCEWithLogitsLoss)
    pos_counts = Y.sum(axis=0)
    neg_counts = Y.shape[0] - pos_counts
    pos_weight = torch.tensor(neg_counts / np.clip(pos_counts, 1, None), dtype=torch.float32).to(device)


Train/val/test split

In [17]:
# Split train/test
if not is_multilabel:
    train_df, temp_df = train_test_split(df, test_size=0.2, stratify=df["y"], random_state=SEED)
    val_df, test_df  = train_test_split(temp_df, test_size=0.5, stratify=temp_df["y"], random_state=SEED)
else:
    # For multi-label, approximate stratification by frequency (simple random split is common)
    train_df, temp_df = train_test_split(df, test_size=0.2, random_state=SEED)
    val_df, test_df  = train_test_split(temp_df, test_size=0.5, random_state=SEED)

Tokenize the Data & Build datasets

In [18]:
train_ds = TextDataset(train_df, is_multilabel=is_multilabel)
val_ds   = TextDataset(val_df,   is_multilabel=is_multilabel)
test_ds  = TextDataset(test_df,  is_multilabel=is_multilabel)

# Set PyTorch format
# train_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])
# test_ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])

In [19]:
# Load config with increased dropout
config = AutoConfig.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    hidden_dropout_prob=0.3,         # affects the hidden layers (default=0.1)
    attention_probs_dropout_prob=0.3 # affects the attention layers (default=0.1)
)

Load Model

In [20]:
if not is_multilabel:
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        config=config
        )
else:
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        config=config,
        problem_type="multi_label_classification"
    )
model.to(device)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


Training Setup

In [21]:
args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=128,
    per_device_eval_batch_size=128,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),   # mixed precision on GPU
    logging_steps=50
)

Define Trainer

In [22]:
trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    class_weights=class_weights,   # from your preprocessing for multi-class
    pos_weight=pos_weight,         # from your preprocessing for multi-label
    is_multilabel=is_multilabel
)

Train the Model

In [23]:
trainer.train()

Epoch,Training Loss,Validation Loss,Macro F1,Accuracy
1,1.032000,0.947407,0.717720,0.749574
2,0.860100,0.888885,0.728606,0.764338
3,0.741800,0.864982,0.732652,0.766894
4,0.711400,0.865199,0.733256,0.766894


TrainOutput(global_step=884, training_loss=0.9399609652040232, metrics={'train_runtime': 1822.2872, 'train_samples_per_second': 61.843, 'train_steps_per_second': 0.485, 'total_flos': 933183876484608.0, 'train_loss': 0.9399609652040232, 'epoch': 4.0})

Evaluate

In [24]:
metrics = trainer.evaluate(test_ds)
metrics

{'eval_loss': 0.8572366833686829,
 'eval_macro_f1': 0.7364708634513526,
 'eval_accuracy': 0.7703009653605906,
 'eval_runtime': 19.8462,
 'eval_samples_per_second': 177.465,
 'eval_steps_per_second': 1.411,
 'epoch': 4.0}

In [25]:
if not is_multilabel:
    preds = np.argmax(trainer.predict(test_ds).predictions, axis=1)
    print(classification_report(test_df["y"], preds, target_names=label_names, digits=3))
else:
    logits = trainer.predict(test_ds).predictions
    probs = 1/(1+np.exp(-logits))
    preds = (probs >= 0.5).astype(int)
    print(classification_report(
        np.vstack(test_ds[:]["labels"]), preds, target_names=label_names, digits=3
    ))

                               precision    recall  f1-score   support

           addiction_disorder      0.798     0.838     0.818       321
             anxiety_disorder      0.746     0.723     0.734       321
             bipolar_disorder      0.814     0.696     0.750       395
             conduct_disorder      0.656     0.629     0.642       264
          depression_disorder      0.531     0.605     0.566       256
  neurodevelopmental_disorder      0.704     0.724     0.714       286
                  no_disorder      0.985     0.988     0.986       721
obsession_compulsion_disorder      0.805     0.754     0.779       236
         personality_disorder      0.707     0.712     0.710       292
               sleep_disorder      0.839     0.862     0.850       217
              stress_disorder      0.525     0.582     0.552       213

                     accuracy                          0.770      3522
                    macro avg      0.737     0.738     0.736      3522
    

Save & load later

In [26]:
trainer.save_model("./mental_health_classifier")
tokenizer.save_pretrained("./mental_health_classifier")

# Reload
from transformers import AutoModelForSequenceClassification, AutoTokenizer
tok = AutoTokenizer.from_pretrained("./mental_health_classifier")
mdl = AutoModelForSequenceClassification.from_pretrained("./mental_health_classifier").to(device)

Make Predictions

In [27]:
print((str(predict(["I’ve been feeling on edge and panicky all week."]))))

['anxiety_disorder']


In [28]:
tdf = pd.read_csv("mental_health_test_data.csv")

tdf.rename(columns={"status": "label"}, inplace=True)
tdf.rename(columns={"statement": "text"}, inplace=True)

# from datasets import Dataset

In [29]:
tdf['label'].value_counts()

label
Normal                  16351
Depression              15404
Suicidal                10653
Anxiety                  3888
Bipolar                  2877
Stress                   2669
Personality disorder     1201
Name: count, dtype: int64

In [30]:
test_label_mapping = {
    "Anxiety": "anxiety_disorder",
    "Stress": "anxiety_disorder",
    "Depression": "depression_disorder",
    "Suicidal": "depression_disorder",
    "Bipolar": "bipolar_disorder",
    "Personality disorder": "personality_disorder",
    "Normal": "no_disorder"
}

tdf['label'] = tdf['label'].replace(test_label_mapping)

tdf['label'].value_counts()

label
depression_disorder     26057
no_disorder             16351
anxiety_disorder         6557
bipolar_disorder         2877
personality_disorder     1201
Name: count, dtype: int64

In [31]:
tdf["text"] = tdf["text"].fillna("").astype(str)

In [32]:
addiction_count = 0
anxiety_count = 0
bipolar_count = 0
conduct_count = 0
depression_count = 0
neurodevelopmental_count = 0
no_count = 0
obsession_compulsion_count = 0
personality_count = 0
sleep_count = 0
stress_count = 0

In [33]:
for string in tdf["text"]:
    match str(predict(string)):
        case "['addiction_disorder']":
            addiction_count += 1
        case "['anxiety_disorder']":
            anxiety_count += 1
        case "['bipolar_disorder']":
            bipolar_count += 1
        case "['conduct_disorder']":
            conduct_count += 1
        case "['depression_disorder']":
            depression_count += 1
        case "['neurodevelopmental_disorder']":
            neurodevelopmental_count += 1
        case "['no_disorder']":
            no_count += 1
        case "['obsession_compulsion_disorder']":
            obsession_compulsion_count += 1
        case "['personality_disorder']":
            personality_count += 1
        case "['sleep_disorder']":
            sleep_count += 1
        case "['stress_disorder']":
            stress_count += 1

In [34]:
print(addiction_count)
print(anxiety_count)
print(bipolar_count)
print(conduct_count)
print(depression_count)
print(neurodevelopmental_count)
print(no_count)
print(obsession_compulsion_count)
print(personality_count)
print(sleep_count)
print(stress_count)

2719
7917
3497
4306
15715
3677
2149
2916
3953
3137
3057


In [ ]:
# label2id = {lbl: i for i, lbl in enumerate(label_names)}

# tdf["y"] = tdf["label"].map(label2id)

In [ ]:
# tdf["y"] = tdf["label"].map(label2id)
# if tdf["y"].isna().any():
#     unknowns = tdf[tdf["y"].isna()]["label"].unique().tolist()
#     raise ValueError(f"Found unknown labels in test set (not in training): {unknowns}")

In [ ]:

# n_eval_ds = Dataset.from_pandas(tdf[["text", "y"]].reset_index(drop=True))

In [ ]:
# def tok(batch):
#     return tokenizer(
#         batch["text"],
#         padding=True,
#         truncation=True,
#         max_length=MAX_LEN,
#     )

# n_eval_ds = n_eval_ds.map(tok, batched=True)
# n_eval_ds = n_eval_ds.remove_columns(["text"])
# n_eval_ds = n_eval_ds.rename_column("y", "labels")
# n_eval_ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

In [ ]:


# pred_out = trainer.predict(n_eval_ds)
# logits = pred_out.predictions
# y_pred = np.argmax(logits, axis=1)
# y_true = np.array(n_eval_ds["labels"])

In [ ]:

# print(classification_report(
#     y_true,
#     y_pred,
#     labels=range(len(label_names)),
#     target_names=label_names,
#     digits=3,
#     zero_division=0
# ))

# from sklearn.metrics import confusion_matrix
# cm = confusion_matrix(y_true, y_pred, labels=range(len(label_names)))
# print("Confusion matrix:\n", cm)
